In [1]:
import subprocess, sys, os

subprocess.run(f"{sys.executable} -m pip install -q "
               "sentence-transformers chromadb", shell=True)

from google.colab import drive
drive.mount('/content/drive')

DOSSIER = "/content/drive/MyDrive/stage-graphrag"
os.makedirs(DOSSIER, exist_ok=True)
print("Contenu du Drive :", os.listdir(DOSSIER))

Mounted at /content/drive
Contenu du Drive : ['corpus.json', 'extraction.jsonl']


In [3]:
import shutil, os
for nom in ("corpus.json", "jeu_de_test.json"):
    if os.path.exists(f"/content/{nom}"):
        shutil.copy(f"/content/{nom}", f"{DOSSIER}/{nom}")
        print("copié :", nom)
print("Sur le Drive :", os.listdir(DOSSIER))

copié : corpus.json
copié : jeu_de_test.json
Sur le Drive : ['corpus.json', 'extraction.jsonl', 'jeu_de_test.json']


In [4]:
import torch
print("GPU disponible :", torch.cuda.is_available())
print("Carte :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "aucune")

GPU disponible : True
Carte : Tesla T4


In [5]:
import json, time, torch
from sentence_transformers import SentenceTransformer

MODELES = {
    "biomed": "pritamdeka/S-PubMedBert-MS-MARCO",
    "general": "sentence-transformers/all-MiniLM-L6-v2",
}

with open(f"{DOSSIER}/corpus.json", encoding="utf-8") as fichier:
    corpus = json.load(fichier)

# Une unité de récupération = un abstract entier (titre + résumé)
unites = [{"pmid": a["pmid"],
           "texte": a["titre"] + "\n\n" + a["resume"]} for a in corpus]
textes = [u["texte"] for u in unites]
print(f"Unités à indexer : {len(unites)}")

index = {}
for nom, chemin in MODELES.items():
    print(f"\n=== {nom} — {chemin} ===")
    debut = time.time()
    modele = SentenceTransformer(chemin, device="cuda")
    vecteurs = modele.encode(
        textes, batch_size=32, convert_to_tensor=True,
        normalize_embeddings=True, show_progress_bar=True,
    )
    index[nom] = vecteurs
    print(f"  dimension : {vecteurs.shape[1]}")
    print(f"  durée     : {time.time() - debut:.0f} s")
    torch.save(vecteurs.cpu(), f"{DOSSIER}/index_{nom}.pt")

with open(f"{DOSSIER}/unites.json", "w", encoding="utf-8") as fichier:
    json.dump(unites, fichier, ensure_ascii=False)
print("\nIndex enregistrés sur le Drive.")

Unités à indexer : 475

=== biomed — pritamdeka/S-PubMedBert-MS-MARCO ===


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'FetchError: Could not fetch resource at https://colab.research.google.com/userdata/get?authuser=0&notebookid=1YgBkIDTwj6a6NeFRpGnjuOTM6x6s21k3&key=HF_TOKEN: 500  '.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

  dimension : 384
  durée     : 10 s

Index enregistrés sur le Drive.


In [6]:
import json, torch
from sentence_transformers import SentenceTransformer

with open(f"{DOSSIER}/jeu_de_test.json", encoding="utf-8") as fichier:
    questions = json.load(fichier)

K = 5
resultats = {}

for nom, chemin in MODELES.items():
    modele = SentenceTransformer(chemin, device="cuda")
    vq = modele.encode([q["question"] for q in questions],
                       convert_to_tensor=True, normalize_embeddings=True)
    scores = vq @ index[nom].T                      # produit scalaire = cosinus
    tops = scores.topk(K, dim=1).indices.cpu().tolist()

    lignes, rappel_total = [], 0
    for q, top in zip(questions, tops):
        recuperes = [unites[i]["pmid"] for i in top]
        attendus = set(q.get("pmids", []))
        trouves = attendus & set(recuperes)
        rappel = len(trouves) / len(attendus) if attendus else None
        if rappel is not None:
            rappel_total += rappel
        lignes.append({"id": q["id"], "type": q["type"],
                       "recuperes": recuperes, "attendus": sorted(attendus),
                       "rappel": rappel})
    resultats[nom] = lignes

    avec = [l for l in lignes if l["rappel"] is not None]
    print(f"\n=== {nom} ===")
    print(f"Rappel@{K} moyen : {rappel_total / len(avec):.2f}")
    for genre in ("single-hop", "pont", "intersection"):
        sous = [l for l in avec if l["type"] == genre]
        if sous:
            m = sum(l["rappel"] for l in sous) / len(sous)
            print(f"  {genre:<14} {m:.2f}  ({len(sous)} questions)")

with open(f"{DOSSIER}/recuperation_vectorielle.json", "w",
          encoding="utf-8") as fichier:
    json.dump(resultats, fichier, ensure_ascii=False, indent=2)
print("\nRésultats enregistrés.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


=== biomed ===
Rappel@5 moyen : 0.55
  single-hop     0.67  (9 questions)
  pont           0.50  (3 questions)
  intersection   0.44  (8 questions)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


=== general ===
Rappel@5 moyen : 0.49
  single-hop     0.67  (9 questions)
  pont           0.33  (3 questions)
  intersection   0.35  (8 questions)

Résultats enregistrés.
